# mBART Pipeline:
Input text -> Encoder -> Decoder -> Generated output

# Explanation:
mBART is a multilingual sequence-to-sequence model based on the BART chitecture.
It is designed to work across multiple languages.
Unlike DistilBART, mBART is not compressed; it is larger and usually slower,
but it can support multilingual input.

# Work:
Input text  
   ↓  
mBART (Transformers + PyTorch)  
   ↓  
Summary / key content  
   ↓  
spaCy  
   ↓  
Entities: contributors, organizations, dates, etc.

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import json
import time
import re
import spacy

c:\Users\joeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load silver data
df = pd.read_json("../Data/silver/doc_01_silver_nlp.json")

# Load mBART
MBART_MODEL = "facebook/mbart-large-50"

tokenizer = AutoTokenizer.from_pretrained(MBART_MODEL, src_lang="en_XX")
model = AutoModelForSeq2SeqLM.from_pretrained(MBART_MODEL)

c:\Users\joeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\joeri\.cache\huggingface\hub\models--facebook--mbart-large-50. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 519/519 [00:00<00:00, 75370.26it/s]
The tied weights map

In [3]:
def summarize_mbart(text,
                    max_input_len=1024,
                    max_output_len=200,
                    min_output_len=40):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len
    )

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            min_length=min_output_len,
            num_beams=4,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [4]:
text = df.loc[0, "cleaned_text"]
summary = summarize_mbart(text)

print(summary)

Delta Water Innovation Hub Project ID: DWIH-2026-01 Start date: 2026-02-01 End date: 2028-12-31 Contact person: Dr. Lisa Vermeer The Delta Water Innovation Hub project focuses on the reuse of treated water streams for agriculture and industrial applications in Zeeland. The project investigates how sensor systems, predictive models and decision-support dashboards can support sustainable water management in coastal regions.


In [5]:
gold_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "mBART",
    "raw_text": text,
    "summary": summary
}

with open("../Data/gold/mBART/mBART.json", "w") as f:
    json.dump(gold_output, f, indent=4)

print("Saved mBART summary to gold layer.")

Saved mBART summary to gold layer.


In [6]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


In [7]:
def evaluate_summary(original_text, summary_text, source_entities=None):
    original_tokens = simple_tokenize(original_text)
    summary_tokens = simple_tokenize(summary_text)

    original_len = len(original_tokens)
    summary_len = len(summary_tokens)

    compression_ratio = summary_len / original_len if original_len > 0 else 0

    preserved_entities = []
    missed_entities = []

    source_entities = source_entities or []
    summary_lower = summary_text.lower()

    for ent in source_entities:
        ent_text = ent[0] if isinstance(ent, (list, tuple)) and len(ent) >= 1 else str(ent)
        ent_text_clean = " ".join(ent_text.split()).strip()

        if ent_text_clean and ent_text_clean.lower() in summary_lower:
            preserved_entities.append(ent_text_clean)
        else:
            missed_entities.append(ent_text_clean)

    total_entities = len(source_entities)
    entity_preservation_score = (
        len(preserved_entities) / total_entities if total_entities > 0 else None
    )

    return {
        "original_token_count": original_len,
        "summary_token_count": summary_len,
        "compression_ratio": compression_ratio,
        "preserved_entities": preserved_entities,
        "missed_entities": missed_entities,
        "entity_preservation_score": entity_preservation_score
    }

In [8]:
def sentence_coverage_score(source_sentences, summary_text, threshold=0.2):
    summary_words = set(simple_tokenize(summary_text))
    covered = 0

    for sent in source_sentences:
        sent_words = set(simple_tokenize(sent))
        if not sent_words:
            continue

        overlap = len(summary_words & sent_words) / len(sent_words)
        if overlap >= threshold:
            covered += 1

    return covered / len(source_sentences) if source_sentences else 0

In [9]:
start = time.time()
summary = summarize_mbart(df.loc[0, "cleaned_text"])
runtime_seconds = time.time() - start

eval_result = evaluate_summary(
    original_text=df.loc[0, "cleaned_text"],
    summary_text=summary,
    source_entities=df.loc[0, "entities"]
)

eval_result["runtime_seconds"] = runtime_seconds
eval_result["sentence_coverage_score"] = sentence_coverage_score(
    df.loc[0, "sentences"], summary
)

evaluation_row = {
    "document_id": df.loc[0, "document_id"],
    "model": "mBART",
    "runtime_seconds": runtime_seconds,
    "original_token_count": eval_result["original_token_count"],
    "summary_token_count": eval_result["summary_token_count"],
    "compression_ratio": eval_result["compression_ratio"],
    "entity_preservation_score": eval_result["entity_preservation_score"],
    "sentence_coverage_score": eval_result["sentence_coverage_score"]
}

with open("../Data/gold/mBART/mBART_evaluation.json", "w") as f:
    json.dump({
        "document_id": df.loc[0, "document_id"],
        "model": "mBART",
        "evaluation": evaluation_row
    }, f, indent=4)

print("Saved mBART evaluation to gold layer.")

Saved mBART evaluation to gold layer.


In [10]:
nlp = spacy.load("en_core_web_sm")

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology"
]

In [11]:
def extract_people(text):
    doc = nlp(text)
    people = [ent.text.strip() for ent in doc.ents if ent.label_ == "PERSON"]
    return sorted(set(people))


In [12]:
def extract_dates(text):
    iso_dates = re.findall(r"\b\d{4}-\d{2}-\d{2}\b", text)
    return sorted(set(iso_dates))


In [13]:
def extract_title(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""

In [14]:
def extract_description(summary, fallback_text):
    if summary and summary.strip():
        return summary.strip()
    return fallback_text[:500].strip()


In [15]:
def extract_research_groups(text, research_groups):
    text_lower = text.lower()
    matches = []

    for group in research_groups:
        if group.lower() in text_lower:
            matches.append(group)

    return sorted(set(matches))


In [16]:
def extract_project_metadata(text, summary=None, document_id=None):
    people = extract_people(text)
    dates = extract_dates(text)
    research_groups = extract_research_groups(text, RESEARCH_GROUPS)

    metadata = {
        "id": document_id or "",
        "title": extract_title(text),
        "description": extract_description(summary, text),
        "contact_person": people[0] if people else "",
        "start_date": dates[0] if len(dates) > 0 else "",
        "end_date": dates[1] if len(dates) > 1 else "",
        "research_groups": research_groups,
        "researchers": people
    }

    return metadata

metadata = extract_project_metadata(
    text=df.loc[0, "cleaned_text"],
    summary=summary,
    document_id=df.loc[0, "document_id"]
)

combined_output = {
    "document_id": df.loc[0, "document_id"],
    "model": "mBART",
    "summary": summary,
    "metadata": metadata,
    "evaluation": {
        "runtime_seconds": runtime_seconds,
        "original_token_count": eval_result["original_token_count"],
        "summary_token_count": eval_result["summary_token_count"],
        "compression_ratio": eval_result["compression_ratio"],
        "entity_preservation_score": eval_result["entity_preservation_score"],
        "sentence_coverage_score": eval_result["sentence_coverage_score"]
    }
}

with open("../Data/gold/mBART/mBART_output.json", "w") as f:
    json.dump(combined_output, f, indent=4)

print("Saved combined mBART output to gold layer.")

Saved combined mBART output to gold layer.
